In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names,file_names_1000):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_13_9_23.xlsx")
    Z = df['Z'].values.reshape(-1, 1)
    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)

    r2_sup_1000 = r2_score(Z, Z_pred_sum/10)
    var = [np.mean((z_pred - Z_pred_sum/10) ** 2) for z_pred in z_preds]
    var_1000 = np.mean(np.array(var))
    bias_1000 = np.mean((Z_pred_sum/10 - Z))

    M = 10  # número de redes
    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds[i]
                f_j = z_preds[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum += cov_ij

    covar_1000 = cov_sum / (M * (M - 1))

    df_list1 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds1 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list1]
    Z_pred_total1 = np.hstack(z_preds1)
    Z_pred_sum1 = np.sum(Z_pred_total1, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("25_model/25_model_13_9_23.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)

    mse_sup1 = np.mean((Z1 - Z_pred_sum1/10) ** 2)

    r2_sup_25 = r2_score(Z1, Z_pred_sum1/10)
    var = [np.mean((z_pred - Z_pred_sum1/10) ** 2) for z_pred in z_preds1]
    var_25 = np.mean(np.array(var))
    bias_25 = np.mean((Z_pred_sum1/10 - Z1))

    M = 10  # número de redes
    cov_sum1 = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds1[i]
                f_j = z_preds1[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum1 += cov_ij

    covar_25 = cov_sum1 / (M * (M - 1))
    
    tabela = pd.DataFrame({
    "Métrica": ["R²", "MSE", "Variância", "Bias", "Covariância"],
    "1024 dados": [r2_sup_1000, mse_sup, var_1000, bias_1000, covar_1000],
    "25 dados": [r2_sup_25, mse_sup1, var_25, bias_25, covar_25]
})

    print(tabela)

    return mse_sup


file_names_1000 =  ["1000_model/1000_model_13_9_23.xlsx", "1000_model/1000_model_14_8_4.xlsx",
              "1000_model/1000_model_23_8_12.xlsx", "1000_model/1000_model_23_8_23.xlsx", 
              "1000_model/1000_model_23_8_6.xlsx", "1000_model/1000_model_27_4_19.xlsx", 
              "1000_model/1000_model_7_9_0.xlsx","1000_model/1000_model_7_9_17.xlsx",
              "1000_model/1000_model_7_9_19.xlsx", "1000_model/1000_model_9_4_4.xlsx"]

file_names = [
    "25_model/25_model_13_9_23.xlsx",
    "25_model/25_model_14_8_4.xlsx",
    "25_model/25_model_23_8_12.xlsx",
    "25_model/25_model_23_8_23.xlsx",
    "25_model/25_model_23_8_6.xlsx",
    "25_model/25_model_27_4_19.xlsx",
    "25_model/25_model_7_9_0.xlsx",
    "25_model/25_model_7_9_17.xlsx",
    "25_model/25_model_7_9_19.xlsx",
    "25_model/25_model_9_4_4.xlsx"
]

result= redes(file_names, file_names_1000)




       Métrica  1024 dados  25 dados
0           R²    0.717702  0.999998
1          MSE    0.196894  0.000001
2    Variância    0.006606  0.000006
3         Bias   -0.011993 -0.000178
4  Covariância    0.689334  0.697428


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

lambda_regs = [1e-6, 5e-6, 1e-5, 1e-3, 1e-2, 5e-5]
lrs = [1e-5, 3e-5 , 1e-2 , 1e-3, 1e-6]
n_epocas_list = [500000,5000000, 10000]

def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    dfZ = pd.read_excel("25_model/25_model_13_9_23.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)

    N = Z.shape[0]
    return Z, z_preds, N


def pesos(Z, z_preds, N, lambda_reg, lr, n_epocas, patience=500, min_delta=1e-2):

    best_erro = np.inf
    best_w = None
    patience_counter = 0

    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas + 1):

        w = np.exp(a) / np.sum(np.exp(a))  # softmax
        yhat = z_preds @ w

        residuo = Z.flatten() - yhat
        mse = np.mean(residuo ** 2)
        mse_pond = mse + lambda_reg * np.sum(a ** 2)

        gmse = (-2.0 / N) * z_preds.T @ residuo
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        a -= lr * grad_a

        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w


def avalia_ensemble(file_names, Z_path, best_w):
    df_list = [pd.read_excel(f) for f in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])

    Z = pd.read_excel(Z_path)['Z'].values.reshape(-1, 1)

    yhat = z_preds @ best_w
    mse = np.mean((Z.flatten() - yhat) ** 2)
    r2 = r2_score(Z, yhat)

    M = z_preds.shape[1]

    var = np.mean([(z_preds[:, i] - yhat) ** 2 for i in range(M)])
    bias = np.mean(yhat - Z.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds[:, i] - z_preds[:, i].mean()) *
                    (z_preds[:, j] - z_preds[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1))

    return mse, r2, var, bias, covar


file_names_1000 =  ["1000_model/1000_model_13_9_23.xlsx", "1000_model/1000_model_14_8_4.xlsx",
              "1000_model/1000_model_23_8_12.xlsx", "1000_model/1000_model_23_8_23.xlsx", 
              "1000_model/1000_model_23_8_6.xlsx", "1000_model/1000_model_27_4_19.xlsx", 
              "1000_model/1000_model_7_9_0.xlsx","1000_model/1000_model_7_9_17.xlsx",
              "1000_model/1000_model_7_9_19.xlsx", "1000_model/1000_model_9_4_4.xlsx"]

file_names_25 = [
   "25_model/25_model_13_9_23.xlsx",
    "25_model/25_model_14_8_4.xlsx",
    "25_model/25_model_23_8_12.xlsx",
    "25_model/25_model_23_8_23.xlsx",
    "25_model/25_model_23_8_6.xlsx",
    "25_model/25_model_27_4_19.xlsx",
    "25_model/25_model_7_9_0.xlsx",
    "25_model/25_model_7_9_17.xlsx",
    "25_model/25_model_7_9_19.xlsx",
    "25_model/25_model_9_4_4.xlsx"
]

# ===============================
# EXECUÇÃO
# ===============================
Z, z_preds, N = redes(file_names_25)

resultados = []

for lambda_reg in lambda_regs:
    for lr in lrs:
        for n_epocas in n_epocas_list:

            best_w = pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

            mse_25, r2_25, var_25, bias_25, covar_25 = avalia_ensemble(
                file_names_25, "25_model/25_model_13_9_23.xlsx", best_w
            )

            mse_1000, r2_1000, _, _, _ = avalia_ensemble(
                file_names_1000, "1000_model/1000_model_13_9_23.xlsx", best_w
            )

            resultados.append({
                "best_pesos": best_w,
                "lambda_reg": lambda_reg,
                "lr": lr,
                "n_epocas": n_epocas,
                "mse_25": mse_25,
                "mse_1000": mse_1000,
                "r2_25": r2_25,
                "r2_1000": r2_1000,
                "var_25": var_25,
                "bias_25": bias_25,
                "covar_25": covar_25
            })


tabela_resultados = pd.DataFrame(resultados)
pd.set_option('display.float_format', '{:.3e}'.format)
tabela_resultados["best_pesos"] = tabela_resultados["best_pesos"].apply(
    lambda w: "[" + ", ".join(f"{x:.6e}" for x in w) + "]"
)


tabela_resultados.to_excel(
    "tabela_resultados_completa.xlsx",
    index=False
)
